# ARTI 402 — Deep Learning
## Lab 7 — Submission (TensorFlow and Keras: Everything You Built, in One Line Each)

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import time
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

print("TensorFlow :", tf.__version__)
print("Keras      :", keras.__version__)

def spiral_data(samples, classes, seed=0):
    rng = np.random.RandomState(seed)
    X = np.zeros((samples * classes, 2))
    y = np.zeros(samples * classes, dtype="uint8")
    for c in range(classes):
        ix = range(samples * c, samples * (c + 1))
        r = np.linspace(0.0, 1, samples)
        t = np.linspace(c * 4, (c + 1) * 4, samples) + rng.randn(samples) * 0.2
        X[ix] = np.c_[r * np.sin(t * 2.5), r * np.cos(t * 2.5)]
        y[ix] = c
    return X, y

X_train, y_train = spiral_data(samples=40,  classes=3, seed=0)
X_val,   y_val   = spiral_data(samples=100, classes=3, seed=2)
X_test,  y_test  = spiral_data(samples=100, classes=3, seed=1)

print("\ntrain:", X_train.shape, " validation:", X_val.shape, " test:", X_test.shape)
print("\nSetup OK")

In [ ]:
from arti402_figures import (draw_overfitting_curves, draw_decision_boundaries,
                             draw_spiral_data)

def history_to_rows(history):
    h = history.history
    return [(i, h["loss"][i], h["accuracy"][i], h["val_loss"][i], h["val_accuracy"][i])
            for i in range(len(h["loss"]))]

print("figure helpers loaded")

In [ ]:
keras.utils.set_random_seed(0)

demo = keras.Sequential([
    layers.Input(shape=(2,)),
    layers.Dense(128, activation="relu"),
    layers.Dense(3, activation="softmax"),
])

demo.compile(optimizer=keras.optimizers.Adam(0.02),
             loss="sparse_categorical_crossentropy",
             metrics=["accuracy"])

demo.summary()

In [ ]:
### Exercise 1 — build it yourself
def build_model(hidden=128, l2=0.0, dropout_rate=0.0, seed=0):
    keras.utils.set_random_seed(seed)
    kernel_reg = regularizers.l2(l2) if l2 > 0 else None

    parts = [layers.Input(shape=(2,))]
    parts.append(layers.Dense(hidden, activation="relu", kernel_regularizer=kernel_reg))

    if dropout_rate > 0:
        parts.append(layers.Dropout(dropout_rate))

    parts.append(layers.Dense(3, activation="softmax"))

    return keras.Sequential(parts)

m = build_model()
assert m.count_params() == 771
assert m.output_shape == (None, 3)

probs = m.predict(X_train[:4], verbose=0)
assert np.allclose(probs.sum(axis=1), 1.0)

m2 = build_model(hidden=32)
assert m2.count_params() == 2 * 32 + 32 + 32 * 3 + 3

m3 = build_model(dropout_rate=0.2)
assert len(m3.layers) == len(m.layers) + 1
print("Exercise 1 passed")

print("\nuntrained predictions (should be near 1/3 each):")
print(probs[:2])

In [ ]:
### Exercise 2 — compile, fit, and look at the curves
model = build_model()

model.compile(optimizer=keras.optimizers.Adam(0.02),
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

t0 = time.time()
history = model.fit(X_train, y_train, epochs=100, batch_size=32, validation_data=(X_val, y_val), verbose=0)
print(f"trained in {time.time() - t0:.0f}s")

train_loss, train_acc = model.evaluate(X_train, y_train, verbose=0)
val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)

print(f"training   : loss {train_loss:.3f}  accuracy {train_acc:.3f}")
print(f"validation : loss {val_loss:.3f}  accuracy {val_acc:.3f}")
print(f"gap        : {train_acc - val_acc:+.3f}")

assert "val_loss" in history.history
assert len(history.history["loss"]) == 100
assert train_acc > 0.6
print("\nExercise 2 passed")

fig = draw_overfitting_curves(history_to_rows(history), "Keras baseline, 100 epochs")
plt.show()

In [ ]:
### Exercise 3 — a regularized model
def compile_and_fit(model, epochs=100, batch_size=32, lr=0.02, callbacks=None, verbose=0):
    model.compile(optimizer=keras.optimizers.Adam(lr),
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    return model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size,
                     validation_data=(X_val, y_val), callbacks=callbacks or [],
                     verbose=verbose)

model_l2 = build_model(l2=5e-4)
hist_l2 = compile_and_fit(model_l2)

l2_train = model_l2.evaluate(X_train, y_train, verbose=0)
l2_val = model_l2.evaluate(X_val, y_val, verbose=0)
print(f"L2 model   : train acc {l2_train[1]:.3f}   val acc {l2_val[1]:.3f}")

assert model_l2.count_params() == 771
assert model_l2.layers[0].kernel_regularizer is not None
assert len(model_l2.losses) > 0
print("\nExercise 3 passed")

penalty = float(sum(model_l2.losses))
print(f"\nthe L2 penalty currently adds {penalty:.4f} to the loss")

In [ ]:
### Exercise 4 — early stopping
early = keras.callbacks.EarlyStopping(monitor="val_loss", patience=25, restore_best_weights=True)

model_es = build_model()
hist_es = compile_and_fit(model_es, epochs=200, callbacks=[early])

epochs_run = len(hist_es.history["loss"])
es_val = model_es.evaluate(X_val, y_val, verbose=0)

print(f"asked for 200 epochs, stopped after {epochs_run}")
print(f"best epoch was {early.best_epoch + 1}, with val loss {early.best:.3f}")
print(f"final validation: loss {es_val[0]:.3f}  accuracy {es_val[1]:.3f}")

assert epochs_run < 200
assert abs(es_val[0] - early.best) < 0.05
print("\nExercise 4 passed")

In [ ]:
train_ds = (tf.data.Dataset
            .from_tensor_slices((X_train, y_train))
            .batch(len(X_train))
            .repeat())

print("dataset element spec:", train_ds.element_spec)
print("\n2000 updates as 2000 epochs : slow (Keras bookkeeping 2000 times)")
print("2000 updates as 100 epochs   : fast (steps_per_epoch=20)")

In [ ]:
model_es.save("spiral_model.keras")
reloaded = keras.models.load_model("spiral_model.keras")

before = model_es.predict(X_val[:5], verbose=0)
after = reloaded.predict(X_val[:5], verbose=0)

print("identical predictions after reload:", np.allclose(before, after))
print("file size:", os.path.getsize("spiral_model.keras"), "bytes")
print("\nreloaded model:")
reloaded.summary()

## Assessment

In [ ]:
### Q1 — The showdown, again
def run_config(name, l2=0.0, dropout_rate=0.0, early_stopping=False,
               steps_per_epoch=20, epochs=100, lr=0.02):
    model = build_model(l2=l2, dropout_rate=dropout_rate)
    model.compile(optimizer=keras.optimizers.Adam(lr),
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    callbacks = []
    if early_stopping:
        callbacks.append(keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=10, restore_best_weights=True))

    hist = model.fit(train_ds, steps_per_epoch=steps_per_epoch, epochs=epochs,
                     validation_data=(X_val, y_val), callbacks=callbacks, verbose=0)

    tr = model.evaluate(X_train, y_train, verbose=0)
    va = model.evaluate(X_val, y_val, verbose=0)
    return model, hist, (tr[1], va[1], va[0])

configs = {
    "baseline":      dict(),
    "L2 (5e-4)":     dict(l2=5e-4),
    "dropout (0.2)": dict(dropout_rate=0.2),
    "early stopping": dict(early_stopping=True),
}

results, models, histories = {}, {}, {}
t0 = time.time()
for name, kwargs in configs.items():
    model, hist, scores = run_config(name, **kwargs)
    models[name], histories[name], results[name] = model, hist, scores

print(f"{'model':<16} {'train acc':>10} {'val acc':>9} {'val loss':>10} {'gap':>8}")
print("-" * 58)
for name, (ta, va, vl) in results.items():
    print(f"{name:<16} {ta:>10.3f} {va:>9.3f} {vl:>10.3f} {ta - va:>+8.3f}")
print(f"\n({time.time() - t0:.0f} seconds)")

assert len(results) == 4
base_ta, base_va, base_vl = results["baseline"]
assert base_ta > 0.9
assert base_ta - base_va > 0.15
for name in ["L2 (5e-4)", "dropout (0.2)", "early stopping"]:
    assert results[name][2] < base_vl
print("Q1 passed")

In [ ]:
### Q2 — Look at it
def predictor(model):
    return lambda points: np.argmax(model.predict(points, verbose=0), axis=1)

fig = draw_decision_boundaries({"baseline": predictor(models["baseline"]),
                                "L2 (5e-4)": predictor(models["L2 (5e-4)"]),
                                "dropout (0.2)": predictor(models["dropout (0.2)"])},
                               X_train, y_train)
plt.show()

fig = draw_overfitting_curves(history_to_rows(histories["baseline"]),
                              "Keras baseline, 2000 updates")
plt.show()